# **Ecology and GIS Synergy: An Example Utilizing an Open Source Dataset**

In this post, I aim to explore the interconnection between ecology and Geographic Information Systems (GIS) through a self-initiated project that analyzes an open-source dataset made available by Stichting Bargerveen. This extensive dataset contains more than 108 thousands entries of different taxa across four countries [link](https://www.gbif.org/dataset/38670987-c8d6-4666-8b42-c263b7c80310#description). 

The initial step in conducting analyses based on free data sources is to clearly define the research objectives. Since such data are often gathered for alternative purposes that are frequently unspecified, it is crucial to establish a methodology to avoid misleading results. In this case, the objective is to evaluate how species community composition is influenced by distance and landscape structure, using Carabidae as a model taxon in The Netherlands.

In [1]:
from shapely.geometry import Polygon,MultiPolygon, Point, LineString
import geopandas as gpd
import pandas as pd
import altair as alt

from string import Template

import numpy as np
import folium
from folium import plugins
from folium.features import DivIcon
from folium.plugins import Fullscreen, Geocoder, MeasureControl,TreeLayerControl,FloatImage,Draw

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [4]:
df_raw = gpd.read_file(r"C:\Users\Luigi\Downloads\0005471-260208012135463\0005471-260208012135463.csv")
df_raw = df_raw[['species','decimalLatitude', 'decimalLongitude','year']]

# Create geometry column from lat/lon
geometry = [Point(xy) for xy in zip(df_raw['decimalLongitude'], df_raw['decimalLatitude'])]
gdf = gpd.GeoDataFrame(df_raw, geometry=geometry, crs="EPSG:4326")


gdf

,species,decimalLatitude,decimalLongitude,year,geometry
0,Oxypselaphus obscurus,53.012671,6.195337,2011,POINT (6.19534 53.01267)
1,Carabus problematicus,52.343501,6.445002,2012,POINT (6.445 52.3435)
2,Harpalus rufipes,52.343501,6.445002,2012,POINT (6.445 52.3435)
3,Calathus fuscipes,52.343501,6.445002,2012,POINT (6.445 52.3435)
4,Amara lunicollis,53.046798,6.311805,2011,POINT (6.3118 53.0468)
...,...,...,...,...,...
27530,Carabus nemoralis,51.403702,5.632001,2011,POINT (5.632 51.4037)
27531,Amara communis,51.403702,5.632001,2011,POINT (5.632 51.4037)
27532,Pterostichus diligens,51.403702,5.632001,2011,POINT (5.632 51.4037)
27533,Amara lunicollis,51.393942,5.620481,2015,POINT (5.62048 51.39394)


Ecology serves as the foundational basis for this analysis. The central questions addressed in this study are: How can we effectively evaluate these influences? How can we compare community composition? While species diversity is the most straightforward and widely utilized method for comparison, this analysis aims to explore the variations in species composition across different sites. Specifically, the focus will be on examining species similarity, species replacement, and nestedness.

The dataset includes point observations of specimens collected from pitfall traps, so my first step was to find the locations of these traps. Next, I organized the observations to create a dataset showing the different species found at each location. To ensure I had a good representation and to reduce noise from yearly changes, I decided to focus on observations from 2023. This year offered enough data points and a good spatial distribution.

In [5]:
df_year = gdf.groupby(['year'],as_index=False)['geometry'].unique()
df_year['number_locations'] = df_year['geometry'].apply(lambda x: len(x))

# Sample data
data = df_year
data.drop('geometry',axis=1,inplace=True)

# Column to highlight
highlight_col = '2023'

# Create bar chart with conditional color
chart = alt.Chart(data).mark_bar().encode(
    x=alt.X('year:N', title=None),
    y=alt.Y('number_locations:Q', title=None),
    color=alt.condition(
        alt.datum.year == highlight_col,  
        alt.value('red'),                  
        alt.value('lightgrey')                 
    )
).properties(
    title=f"Number locations per year",
    width=400,
    height=300
)

chart.show()

alt.Chart(...)

In [6]:
gdf_2023 = gdf[gdf['year']=='2023']
df_sp = gdf_2023.groupby(['geometry'],as_index=False)['species'].unique()
df_sp['site'] = list(range(1,len(df_sp)+1))
gdf_sp = gpd.GeoDataFrame(df_sp,geometry=df_sp.geometry)



m = folium.Map(location=[gdf_2023.geometry.y.mean(), gdf_2023.geometry.x.mean()],zoom_start=7, control_scale=True)



for point in gdf_2023['geometry'].unique():
    gdf_temp = gdf_2023[gdf_2023['geometry']==point]
    html_table = gdf_temp.groupby(['species']).size()\
    .to_frame("Number of specimens")\
    .to_html(classes="table table-striped", index=True)


    #___
    html_content = """
    <style>
    table, th, td {
          border: 1px solid black;
          border-radius: 1px;
          padding-top: 5px;
          padding-bottom: 10px;
          padding-left: 15px;
          padding-right: 20px;
        }
        
    h1 {
        text-align: center;
        color: Green;
        font-family: Arial, sans-serif;
    }
    
    summary {
        background-color: #007BFF;
        color: white;
        padding: 5px;
        border-radius: 4px;
        cursor: pointer;
        font-family: Arial, sans-serif;
    }
    
    summary:hover {
        background-color: #0056b3;
        margin: 5px 0;
    }
    
    </style>
    
    
    
    <details>
      <summary>Table</summary>
        <table>$html_table</table>
    </details>
    
    """
    
    template = Template(html_content)
    html_content = template.substitute(html_table=html_table)

    #___

    
    # Wrap HTML in an IFrame so Folium can render it properly
    iframe = folium.IFrame(html=html_content, width=650, height=650,)
    popup=folium.Popup(iframe,width='100%',sticky=True)

    folium.Marker(location=[point.y,point.x],
                  popup = popup,
                  tooltip='click here'
                 ).add_to(m)


    
folium.plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m)

# <h1>$site</h1>
# ,site=gdf_temp['site'].unique()[0]

m